<a href="https://colab.research.google.com/github/zhangwiki86-soton/Python-Files-for-Practice/blob/main/Farm%20DE%20draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
from datetime import datetime

In [2]:


DATABASE_NAME = "farm_management.db"


def create_database():
    connection = sqlite3.connect(DATABASE_NAME)

    # Enforce foreign-key constraints in SQLite.
    connection.execute("PRAGMA foreign_keys = ON")

    cursor = connection.cursor()

    # ---------------------------------------------------------
    # Drop existing tables so the program can be run repeatedly
    # ---------------------------------------------------------

    cursor.executescript("""
        DROP TABLE IF EXISTS Farm_Initiative;
        DROP TABLE IF EXISTS Resource_Application;
        DROP TABLE IF EXISTS Crop_Cycle;
        DROP TABLE IF EXISTS Sustainability_Initiative;
        DROP TABLE IF EXISTS Resource_Type;
        DROP TABLE IF EXISTS Crop;
        DROP TABLE IF EXISTS Soil;
        DROP TABLE IF EXISTS Farm;
        DROP TABLE IF EXISTS Water_Source;
    """)

    # ---------------------------------------------------------
    # Create tables
    # ---------------------------------------------------------

    cursor.executescript("""
        CREATE TABLE Water_Source (
            water_source_id INTEGER PRIMARY KEY,
            source_type TEXT NOT NULL UNIQUE
        );

        CREATE TABLE Farm (
            farm_id INTEGER PRIMARY KEY,
            farm_location TEXT NOT NULL,
            water_source_id INTEGER NOT NULL,
            FOREIGN KEY (water_source_id)
                REFERENCES Water_Source(water_source_id)
        );

        CREATE TABLE Soil (
            soil_id INTEGER PRIMARY KEY,
            ph_level REAL NOT NULL,
            nitrogen_level REAL NOT NULL,
            phosphorus_level REAL NOT NULL,
            potassium_level REAL NOT NULL
        );

        CREATE TABLE Crop (
            crop_id INTEGER PRIMARY KEY,
            crop_name TEXT NOT NULL UNIQUE
        );

        CREATE TABLE Resource_Type (
            resource_type_id INTEGER PRIMARY KEY,
            resource_type TEXT NOT NULL UNIQUE
        );

        CREATE TABLE Sustainability_Initiative (
            initiative_id INTEGER PRIMARY KEY,
            initiative_description TEXT NOT NULL UNIQUE,
            date_initiated DATE NOT NULL,
            expected_impact TEXT NOT NULL
        );

        CREATE TABLE Crop_Cycle (
            crop_cycle_id INTEGER PRIMARY KEY AUTOINCREMENT,
            farm_id INTEGER NOT NULL,
            crop_id INTEGER NOT NULL,
            soil_id INTEGER NOT NULL,
            planting_date DATE NOT NULL,
            harvest_date DATE NOT NULL,
            crop_yield REAL NOT NULL,
            labour_hours REAL NOT NULL,
            environmental_impact_score INTEGER NOT NULL,

            FOREIGN KEY (farm_id)
                REFERENCES Farm(farm_id),

            FOREIGN KEY (crop_id)
                REFERENCES Crop(crop_id),

            FOREIGN KEY (soil_id)
                REFERENCES Soil(soil_id),

            CHECK (harvest_date >= planting_date),
            CHECK (crop_yield >= 0),
            CHECK (labour_hours >= 0),
            CHECK (environmental_impact_score >= 0)
        );

        CREATE TABLE Resource_Application (
            resource_application_id INTEGER PRIMARY KEY AUTOINCREMENT,
            crop_cycle_id INTEGER NOT NULL,
            resource_type_id INTEGER NOT NULL,
            resource_quantity REAL NOT NULL,
            date_of_application DATE NOT NULL,

            FOREIGN KEY (crop_cycle_id)
                REFERENCES Crop_Cycle(crop_cycle_id),

            FOREIGN KEY (resource_type_id)
                REFERENCES Resource_Type(resource_type_id),

            CHECK (resource_quantity >= 0)
        );

        CREATE TABLE Farm_Initiative (
            farm_id INTEGER NOT NULL,
            initiative_id INTEGER NOT NULL,

            PRIMARY KEY (farm_id, initiative_id),

            FOREIGN KEY (farm_id)
                REFERENCES Farm(farm_id),

            FOREIGN KEY (initiative_id)
                REFERENCES Sustainability_Initiative(initiative_id)
        );
    """)

    # ---------------------------------------------------------
    # Insert water sources
    # ---------------------------------------------------------

    water_sources = [
        (1, "River"),
        (2, "Borehole"),
        (3, "Rainwater"),
        (4, "Well")
    ]

    cursor.executemany("""
        INSERT INTO Water_Source
            (water_source_id, source_type)
        VALUES (?, ?)
    """, water_sources)

    # ---------------------------------------------------------
    # Insert farms
    # ---------------------------------------------------------

    farms = [
        (1, "South Farm, Kent", 1),
        (2, "Green Acres, Essex", 2),
        (3, "Sunny Fields, Hampshire", 3),
        (4, "Hilltop Farm, Yorkshire", 4),
        (5, "Riverbend Farm, Cornwall", 1)
    ]

    cursor.executemany("""
        INSERT INTO Farm
            (farm_id, farm_location, water_source_id)
        VALUES (?, ?, ?)
    """, farms)

    # ---------------------------------------------------------
    # Insert soil records
    # ---------------------------------------------------------

    soils = [
        (1, 6.5, 50, 20, 180),
        (2, 6.8, 40, 25, 160),
        (3, 6.2, 30, 15, 150),
        (4, 6.4, 45, 22, 175),
        (5, 6.7, 55, 28, 200)
    ]

    cursor.executemany("""
        INSERT INTO Soil
            (
                soil_id,
                ph_level,
                nitrogen_level,
                phosphorus_level,
                potassium_level
            )
        VALUES (?, ?, ?, ?, ?)
    """, soils)

    # ---------------------------------------------------------
    # Insert crops
    # ---------------------------------------------------------

    crops = [
        (101, "Wheat"),
        (102, "Barley"),
        (201, "Corn"),
        (202, "Soybeans"),
        (301, "Potatoes"),
        (302, "Carrots"),
        (401, "Apples"),
        (402, "Pears"),
        (501, "Tomatoes"),
        (502, "Lettuce")
    ]

    cursor.executemany("""
        INSERT INTO Crop
            (crop_id, crop_name)
        VALUES (?, ?)
    """, crops)

    # ---------------------------------------------------------
    # Insert resource types
    # ---------------------------------------------------------

    resource_types = [
        (1, "Water"),
        (2, "Fertilizer")
    ]

    cursor.executemany("""
        INSERT INTO Resource_Type
            (resource_type_id, resource_type)
        VALUES (?, ?)
    """, resource_types)

    # ---------------------------------------------------------
    # Insert sustainability initiatives
    # ---------------------------------------------------------

    initiatives = [
        (
            1,
            "Organic Farming",
            "2023-01-01",
            "Increase in yield"
        ),
        (
            2,
            "Crop Rotation",
            "2023-02-15",
            "Improved soil quality"
        ),
        (
            3,
            "Water Conservation",
            "2023-03-01",
            "Reduced water usage"
        ),
        (
            4,
            "Soil Health Improvement",
            "2023-01-20",
            "Enhanced nutrient retention"
        ),
        (
            5,
            "Pesticide Reduction",
            "2023-02-10",
            "Less chemical runoff"
        )
    ]

    cursor.executemany("""
        INSERT INTO Sustainability_Initiative
            (
                initiative_id,
                initiative_description,
                date_initiated,
                expected_impact
            )
        VALUES (?, ?, ?, ?)
    """, initiatives)

    # ---------------------------------------------------------
    # Insert crop cycles
    #
    # One record represents one crop being grown on one farm.
    # ---------------------------------------------------------

    crop_cycles = [
        (
            1, 101, 1,
            "2023-03-15", "2023-08-15",
            3000, 150, 4
        ),
        (
            1, 102, 1,
            "2023-03-16", "2023-08-20",
            2800, 120, 4
        ),
        (
            2, 201, 2,
            "2023-04-10", "2023-09-15",
            1500, 200, 3
        ),
        (
            2, 202, 2,
            "2023-04-11", "2023-09-20",
            1200, 180, 3
        ),
        (
            3, 301, 3,
            "2023-03-20", "2023-07-15",
            2000, 160, 5
        ),
        (
            3, 302, 3,
            "2023-03-21", "2023-07-20",
            2200, 170, 5
        ),
        (
            4, 401, 4,
            "2023-04-05", "2023-09-10",
            1800, 140, 2
        ),
        (
            4, 402, 4,
            "2023-04-06", "2023-09-15",
            1600, 130, 2
        ),
        (
            5, 501, 5,
            "2023-03-25", "2023-08-10",
            2500, 190, 4
        ),
        (
            5, 502, 5,
            "2023-03-26", "2023-08-15",
            2400, 175, 4
        )
    ]

    cursor.executemany("""
        INSERT INTO Crop_Cycle
            (
                farm_id,
                crop_id,
                soil_id,
                planting_date,
                harvest_date,
                crop_yield,
                labour_hours,
                environmental_impact_score
            )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, crop_cycles)

    # ---------------------------------------------------------
    # Insert resource applications
    #
    # crop_cycle_id corresponds to the order in which the
    # ten crop-cycle records were inserted.
    # ---------------------------------------------------------

    resource_applications = [
        (1, 1, 1000, "2023-03-10"),
        (2, 2, 200, "2023-03-12"),
        (3, 1, 800, "2023-04-05"),
        (4, 2, 150, "2023-04-06"),
        (5, 1, 1200, "2023-03-18"),
        (6, 2, 300, "2023-03-19"),
        (7, 1, 900, "2023-04-02"),
        (8, 2, 250, "2023-04-03"),
        (9, 1, 1100, "2023-03-22"),
        (10, 2, 180, "2023-03-24")
    ]

    cursor.executemany("""
        INSERT INTO Resource_Application
            (
                crop_cycle_id,
                resource_type_id,
                resource_quantity,
                date_of_application
            )
        VALUES (?, ?, ?, ?)
    """, resource_applications)

    # ---------------------------------------------------------
    # Insert farm/initiative relationships
    #
    # Each farm has the initiative shown in the original data.
    # ---------------------------------------------------------

    farm_initiatives = [
        (1, 1),
        (2, 2),
        (3, 3),
        (4, 4),
        (5, 5)
    ]

    cursor.executemany("""
        INSERT INTO Farm_Initiative
            (farm_id, initiative_id)
        VALUES (?, ?)
    """, farm_initiatives)

    # Save changes
    connection.commit()

    # ---------------------------------------------------------
    # Basic verification
    # ---------------------------------------------------------

    tables = [
        "Farm",
        "Water_Source",
        "Soil",
        "Crop",
        "Resource_Type",
        "Sustainability_Initiative",
        "Crop_Cycle",
        "Resource_Application",
        "Farm_Initiative"
    ]

    print("Database created successfully.")
    print(f"Database file: {DATABASE_NAME}")
    print()

    for table in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"{table}: {count} record(s)")

    connection.close()


if __name__ == "__main__":
    create_database()

Database created successfully.
Database file: farm_management.db

Farm: 5 record(s)
Water_Source: 4 record(s)
Soil: 5 record(s)
Crop: 10 record(s)
Resource_Type: 2 record(s)
Sustainability_Initiative: 5 record(s)
Crop_Cycle: 10 record(s)
Resource_Application: 10 record(s)
Farm_Initiative: 5 record(s)
